In [1]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle



In [2]:
## Load the dataset 
data=pd.read_csv("Churn_Modeling.csv")
data.head()

,CustomerID,Gender,Age,Tenure_Months,MonthlyCharges,TotalCharges,ContractType,PaymentMethod,InternetService,TechSupport,OnlineBackup,Churn
0,CUST0001,Male,34,11,768.54,8971.54,Month-to-Month,UPI,Fiber Optic,No,No,1
1,CUST0002,Female,26,56,1638.12,37887.15,Two Year,Net Banking,Fiber Optic,No,Yes,0
2,CUST0003,Male,50,67,1957.22,112268.00,Month-to-Month,UPI,Fiber Optic,No,No,1
3,CUST0004,Male,37,29,931.05,102156.29,One Year,Cash,DSL,No,Yes,0
4,CUST0005,Male,30,9,1029.11,117837.73,Month-to-Month,UPI,DSL,No,No,1


In [3]:
##Preprocess the data
### Drop irrelevent columns
data=data.drop(['CustomerID','Gender','MonthlyCharges'],axis=1)
data

,Age,Tenure_Months,TotalCharges,ContractType,PaymentMethod,InternetService,TechSupport,OnlineBackup,Churn
0,34,11,8971.54,Month-to-Month,UPI,Fiber Optic,No,No,1
1,26,56,37887.15,Two Year,Net Banking,Fiber Optic,No,Yes,0
2,50,67,112268.00,Month-to-Month,UPI,Fiber Optic,No,No,1
3,37,29,102156.29,One Year,Cash,DSL,No,Yes,0
4,30,9,117837.73,Month-to-Month,UPI,DSL,No,No,1
...,...,...,...,...,...,...,...,...,...
995,42,30,58384.75,Month-to-Month,Debit Card,DSL,Yes,Yes,0
996,26,24,83409.76,One Year,Credit Card,Fiber Optic,No,No,0
997,21,30,66555.10,One Year,Credit Card,Fiber Optic,No,Yes,0
998,31,7,26341.94,Month-to-Month,Net Banking,NaN,No,No,1


In [4]:
##Encode one categorial varibales
label_encoder_contractType=LabelEncoder()
data['ContractType']=label_encoder_contractType.fit_transform(data['ContractType'])
data

,Age,Tenure_Months,TotalCharges,ContractType,PaymentMethod,InternetService,TechSupport,OnlineBackup,Churn
0,34,11,8971.54,0,UPI,Fiber Optic,No,No,1
1,26,56,37887.15,2,Net Banking,Fiber Optic,No,Yes,0
2,50,67,112268.00,0,UPI,Fiber Optic,No,No,1
3,37,29,102156.29,1,Cash,DSL,No,Yes,0
4,30,9,117837.73,0,UPI,DSL,No,No,1
...,...,...,...,...,...,...,...,...,...
995,42,30,58384.75,0,Debit Card,DSL,Yes,Yes,0
996,26,24,83409.76,1,Credit Card,Fiber Optic,No,No,0
997,21,30,66555.10,1,Credit Card,Fiber Optic,No,Yes,0
998,31,7,26341.94,0,Net Banking,NaN,No,No,1


In [5]:
##Onehot encode 'PaymentMethod
from sklearn.preprocessing import OneHotEncoder
onehot_encoder=OneHotEncoder()
pay_encoder = onehot_encoder.fit_transform(data[['PaymentMethod']])
pay_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1000 stored elements and shape (1000, 5)>

In [6]:
pay_encoder.toarray()

array([[0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1.],
       ...,
       [0., 1., 0., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 1.]])

In [7]:
onehot_encoder.get_feature_names_out(['PaymentMethod'])


array(['PaymentMethod_Cash', 'PaymentMethod_Credit Card',
       'PaymentMethod_Debit Card', 'PaymentMethod_Net Banking',
       'PaymentMethod_UPI'], dtype=object)

In [8]:
pay_encoded_df=pd.DataFrame(pay_encoder.toarray(),columns=onehot_encoder.get_feature_names_out(['PaymentMethod']))
pay_encoded_df

,PaymentMethod_Cash,PaymentMethod_Credit Card,PaymentMethod_Debit Card,PaymentMethod_Net Banking,PaymentMethod_UPI
0,0.0,0.0,0.0,0.0,1.0
1,0.0,0.0,0.0,1.0,0.0
2,0.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...
995,0.0,0.0,1.0,0.0,0.0
996,0.0,1.0,0.0,0.0,0.0
997,0.0,1.0,0.0,0.0,0.0
998,0.0,0.0,0.0,1.0,0.0


In [9]:
## Combine one hot encoder columns with the original data
data=pd.concat([data.drop('PaymentMethod',axis=1),pay_encoded_df],axis=1)
data.head()

,Age,Tenure_Months,TotalCharges,ContractType,InternetService,TechSupport,OnlineBackup,Churn,PaymentMethod_Cash,PaymentMethod_Credit Card,PaymentMethod_Debit Card,PaymentMethod_Net Banking,PaymentMethod_UPI
0,34,11,8971.54,0,Fiber Optic,No,No,1,0.0,0.0,0.0,0.0,1.0
1,26,56,37887.15,2,Fiber Optic,No,Yes,0,0.0,0.0,0.0,1.0,0.0
2,50,67,112268.00,0,Fiber Optic,No,No,1,0.0,0.0,0.0,0.0,1.0
3,37,29,102156.29,1,DSL,No,Yes,0,1.0,0.0,0.0,0.0,0.0
4,30,9,117837.73,0,DSL,No,No,1,0.0,0.0,0.0,0.0,1.0


In [10]:
##Save the encoders and scaler
with open('label_encoder_contractType.pkl','wb') as file:
    pickle.dump(label_encoder_contractType,file)

with open('onehot_encoder_paymentMethod.pkl' , 'wb') as file:
    pickle.dump(onehot_encoder,file)

In [11]:
data.head()

,Age,Tenure_Months,TotalCharges,ContractType,InternetService,TechSupport,OnlineBackup,Churn,PaymentMethod_Cash,PaymentMethod_Credit Card,PaymentMethod_Debit Card,PaymentMethod_Net Banking,PaymentMethod_UPI
0,34,11,8971.54,0,Fiber Optic,No,No,1,0.0,0.0,0.0,0.0,1.0
1,26,56,37887.15,2,Fiber Optic,No,Yes,0,0.0,0.0,0.0,1.0,0.0
2,50,67,112268.00,0,Fiber Optic,No,No,1,0.0,0.0,0.0,0.0,1.0
3,37,29,102156.29,1,DSL,No,Yes,0,1.0,0.0,0.0,0.0,0.0
4,30,9,117837.73,0,DSL,No,No,1,0.0,0.0,0.0,0.0,1.0


In [12]:
##Divide the dataset into indepent and dependent features
X=data.drop('Churn', axis=1)
y=data['Churn']

# Encode remaining categorical columns
from sklearn.preprocessing import LabelEncoder

le_internet = LabelEncoder()
data['InternetService'] = le_internet.fit_transform(data['InternetService'])

le_tech = LabelEncoder()
data['TechSupport'] = le_tech.fit_transform(data['TechSupport'])

le_backup = LabelEncoder()
data['OnlineBackup'] = le_backup.fit_transform(data['OnlineBackup'])

##Split the dataset into training and testing sets
X_train, X_test, y_train, y_test=train_test_split(X,y,test_size=0.2, random_state=42)

##Scale the features
scaler=StandardScaler()

In [13]:
X_train

,Age,Tenure_Months,TotalCharges,ContractType,InternetService,TechSupport,OnlineBackup,PaymentMethod_Cash,PaymentMethod_Credit Card,PaymentMethod_Debit Card,PaymentMethod_Net Banking,PaymentMethod_UPI
29,28,56,93829.01,0,Fiber Optic,No,Yes,0.0,0.0,1.0,0.0,0.0
535,24,39,44188.69,0,NaN,No,Yes,0.0,0.0,1.0,0.0,0.0
695,68,7,35606.22,0,NaN,No,No,0.0,0.0,1.0,0.0,0.0
557,67,5,53288.71,0,DSL,Yes,No,0.0,0.0,1.0,0.0,0.0
836,62,47,81664.30,0,DSL,No,No,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
106,32,26,119553.34,1,Fiber Optic,Yes,Yes,0.0,0.0,1.0,0.0,0.0
270,47,66,87892.32,0,Fiber Optic,No,Yes,0.0,0.0,0.0,0.0,1.0
860,48,56,52862.12,2,Fiber Optic,No,No,0.0,0.0,0.0,0.0,1.0
435,27,37,68553.40,1,Fiber Optic,No,No,0.0,0.0,0.0,0.0,1.0


In [14]:
with open ('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)


ANN IMPLEMENTATION

In [15]:
!pip install tensorflow

In [16]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime


e:\New folder\envs\ann_env\Lib\site-packages\h5py\__init__.py:36: UserWarning: h5py is running against HDF5 1.14.6 when it was built against 1.14.5, this may cause problems
  _warn(("h5py is running against HDF5 {0} when it was built against {1}, "


In [17]:
(X_train.shape[1],)

(12,)

In [18]:
##Build our ANN Model
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)), ## HL1 Connected with input layer 
    Dense(32, activation='relu'), ## HL2
    Dense(1, activation='sigmoid') ## Output layer
])

e:\New folder\envs\ann_env\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()
loss

<LossFunctionWrapper(<function binary_crossentropy at 0x00000203A412C540>, kwargs={'from_logits': False, 'label_smoothing': 0.0, 'axis': -1})>

In [21]:
## Compile the model
model.compile(optimizer=opt,loss="binary_crossentropy",metrics=['accuracy'])

##Divide the dataset into independent and dependent features
# Encode remaining categorical columns FIRST
le_internet = LabelEncoder()
data['InternetService'] = le_internet.fit_transform(data['InternetService'].fillna('None'))

le_tech = LabelEncoder()
data['TechSupport'] = le_tech.fit_transform(data['TechSupport'])

le_backup = LabelEncoder()
data['OnlineBackup'] = le_backup.fit_transform(data['OnlineBackup'])

# NOW split X and y AFTER encoding
X = data.drop('Churn', axis=1)
y = data['Churn']

##Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

##Scale the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [22]:
## Set up the Tensorboard
from tensorflow.keras.callbacks import EarlyStopping , TensorBoard
log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard()

In [23]:
## Set up Early Stopping 
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [24]:
from tensorflow.keras.callbacks import TensorBoard
import datetime

log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

tensorboard_cb = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1
)


In [25]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=32,
    callbacks=[tensorboard_cb]
)


Epoch 1/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 49ms/step - accuracy: 0.5525 - loss: 0.6812 - val_accuracy: 0.6400 - val_loss: 0.6473
Epoch 2/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6550 - loss: 0.6385 - val_accuracy: 0.6200 - val_loss: 0.6460
Epoch 3/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6650 - loss: 0.6201 - val_accuracy: 0.6200 - val_loss: 0.6485
Epoch 4/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6625 - loss: 0.6130 - val_accuracy: 0.6250 - val_loss: 0.6473
Epoch 5/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6787 - loss: 0.6020 - val_accuracy: 0.5900 - val_loss: 0.6858
Epoch 6/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6812 - loss: 0.5906 - val_accuracy: 0.5950 - val_loss: 0.6590
Epoch 7/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6950 - loss: 0.5755 - val_accuracy: 0.6000 - val_loss: 0.7267
Epoch 8/20
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6988 - loss: 0.5686 - val_accuracy: 0.6000 - v

In [26]:
model.save('model.h5')

In [27]:
%load_ext tensorboard
%tensorboard --logdir logs/fit --port 6007

Reusing TensorBoard on port 6007 (pid 24924), started 23:49:58 ago. (Use '!kill 24924' to kill it.)

In [28]:
import sys
print(sys.executable)

e:\New folder\envs\ann_env\python.exe


In [29]:
!pip install setuptools tensorboard


In [32]:
%load_ext tensorboard
%tensorboard --logdir logs/fit

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 6006 (pid 26984), started 4 days, 14:28:44 ago. (Use '!kill 26984' to kill it.)

In [33]:
!taskkill /F /PID 24924


ERROR: The process "24924" not found.
